<a href="https://colab.research.google.com/github/GilangAngga12/Pemrosesan-teks-praktik/blob/main/1%20oktober.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%pip install -q "scrapling[fetchers]" pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.5/41.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.8/761.8 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.2/48.2 MB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.5/180.5 kB 13.8 MB/s eta 0:00:00


In [2]:
import re
import time
from datetime import datetime
from urllib.parse import urljoin

import pandas as pd
from scrapling.fetchers import Fetcher

In [3]:
# Konfigurasi

MAX_ARTICLES_PER_SOURCE = 30
REQUEST_DELAY = 1.0

OUTPUT_FILE = "news.csv"

In [4]:
# Pagination handling
PAGINATION_SELECTORS = [
    "a[rel='next']",
    "a.next",
    "a:contains('Selanjutnya')",
    "a:contains('Berikutnya')",
]

In [5]:
# Keywords
CATEGORY = "otomotif"

KEYWORDS = {
    "otomotif": [
        "otomotif",
        "mobil",
        "motor",
        "kendaraan",
        "mobil listrik",
        "motor listrik",
        "toyota",
        "honda",
        "yamaha",
        "hyundai",
        "suzuki",
        "wuling",
        "byd",
        "daihatsu",
        "mitsubishi",
    ]
}

In [6]:
# konfigurasi sumber
SOURCES = {
    "inews": {
        "otomotif": "https://www.inews.id/otomotif"
    },

    # "kompas": {
    #     "otomotif": "https://otomotif.kompas.com/"
    # }
}

In [7]:
# Pagination handling
PAGINATION_SELECTORS = [
    "a[rel='next']",
    "a.next",
    "a:contains('Selanjutnya')",
    "a:contains('Berikutnya')",
]
MAX_PAGES_PER_ARTICLE = None   # set to int to limit pages per article

def fetch_all_pages(start_url, site):
    """Ambil semua halaman artikel yang dipaginasikan dan gabungkan isinya."""
    cur = start_url
    full = ""
    pages = 0
    while True:
        page = fetch(cur, site.get('mode', 'http'))
        if page.status != 200:
            break
        part = extract_content(page, site.get('content_selectors', []))
        if part:
            full += ("\n\n" if full else "") + part
        nxt = None
        for sel in PAGINATION_SELECTORS:
            nxt = page.css(f"{sel}::attr(href)").get()
            if nxt:
                break
        if not nxt:
            break
        cur = urljoin(cur, nxt)
        pages += 1
        if MAX_PAGES_PER_ARTICLE is not None and pages >= MAX_PAGES_PER_ARTICLE:
            break
        time.sleep(REQUEST_DELAY)
    return full

In [8]:
# utility

def match_keywords(title, content, category):
    """
    Mencari keyword yang muncul pada judul atau isi artikel.
    """

    text = f"{title} {content}".lower()

    matched = []

    for keyword in KEYWORDS.get(category, []):

        if keyword.lower() in text:
            matched.append(keyword)

    return matched

In [9]:
# kategori
def get_article_links(page, source):
    """
    Mengambil URL artikel dari halaman kategori.
    """

    links = []

    if source == "inews":

        # Semua link yang berada di halaman.
        raw_links = page.css("a::attr(href)").getall()

        for url in raw_links:

            if not url:
                continue

            if "/read/" in url:

                links.append(url)

            elif "/otomotif/" in url:

                links.append(url)


    elif source == "kompas":

        raw_links = page.css("a::attr(href)").getall()

        for url in raw_links:

            if not url:
                continue

            if "kompas.com" in url:

                if "/otomotif/" in url or "/bola/" in url:
                    links.append(url)

    return links

In [11]:
# exstrack artikel
def scrape_article(url, source, category):
    """
    Mengambil detail artikel.
    """

    print(f"  -> Scraping: {url}")

    try:

        page = Fetcher.get(
            url,
            stealthy_headers=True
        )

    except Exception as e:

        print(f"     ERROR request: {e}")

        return None

    # TITLE

    title = ""

    title_candidates = [
        "h1",
        "meta[property='og:title']::attr(content)",
        "title::text",
    ]

    for selector in title_candidates:

        try:

            result = page.css(selector).get()

            if result:

                # title = clean_text(
                #     result if isinstance(result, str)
                #     else result.text
                # )
                title = result if isinstance(result, str) else result.text

                if title:
                    break

        except Exception:
            continue

    # DATE

    date = ""

    date_candidates = [
        "time::attr(datetime)",
        "time::text",
        "meta[property='article:published_time']::attr(content)",
    ]

    for selector in date_candidates:

        try:

            result = page.css(selector).get()

            if result:

                # date = clean_text(
                #     result if isinstance(result, str)
                #     else result.text
                # )
                date = result if isinstance(result, str) else result.text

                if date:
                    break

        except Exception:
            continue

    # CONTENT

    paragraphs = []

    content_selectors = [
        "article p::text",
        ".article-content p::text",
        ".read__content p::text",
        ".detail__body-text p::text",
        ".content p::text",
    ]

    for selector in content_selectors:

        try:

            paragraphs = page.css(selector).getall()

            if paragraphs:
                break

        except Exception:
            continue

    content = " ".join(
        # clean_text(p)
        p for p in paragraphs
        if p
    )

    # KEYWORD MATCHING

    matched_keywords = match_keywords(
        title,
        content,
        category
    )

    if not matched_keywords:

        return None

    # RESULT

    return {
        "source": source,
        "category": category,
        "title": title,
        "url": url, # normalize_url(url),
        "date": date,
        "keywords": ", ".join(matched_keywords),
        "content": content,
        "scraped_at": datetime.now().isoformat(),
    }

In [13]:
import re
import time
from datetime import datetime
from urllib.parse import urljoin

import pandas as pd
from scrapling.fetchers import Fetcher

# main
def scrape_category(source, category, category_url):
    """
    Scrapes articles from a given category URL.
    """
    print(f"==> Scraping category '{category}' from {source}: {category_url}")

    articles = []
    current_page_url = category_url
    articles_count = 0
    page_num = 1

    while current_page_url and articles_count < MAX_ARTICLES_PER_SOURCE:
        print(f"  -> Fetching category page {page_num}: {current_page_url}")
        try:
            page = Fetcher.get(current_page_url, stealthy_headers=True)
        except Exception as e:
            print(f"     ERROR fetching category page: {e}")
            break

        if page.status != 200:
            print(f"     ERROR: Received status code {page.status} for {current_page_url}")
            break

        article_links = get_article_links(page, source)
        print(f"     Found {len(article_links)} potential article links.")

        for link in article_links:
            if articles_count >= MAX_ARTICLES_PER_SOURCE:
                break

            full_url = urljoin(category_url, link)
            article_data = scrape_article(full_url, source, category)
            if article_data:
                articles.append(article_data)
                articles_count += 1
            time.sleep(REQUEST_DELAY)

        # Find next page for pagination
        next_page_url = None
        for sel in PAGINATION_SELECTORS:
            next_page_url = page.css(f"{sel}::attr(href)").get()
            if next_page_url:
                break

        if next_page_url:
            current_page_url = urljoin(category_url, next_page_url)
            page_num += 1
            time.sleep(REQUEST_DELAY)
        else:
            current_page_url = None # No more pages

    print(f"<== Finished scraping category '{category}'. Scraped {len(articles)} articles.")
    return articles

def main():

    all_results = []

    for source_name, categories in SOURCES.items():

        for category_name, url in categories.items():

            results = scrape_category(
                source=source_name,
                category=category_name,
                category_url=url,
            )

            all_results.extend(results)

    # DATAFRAME

    df = pd.DataFrame(all_results)

    if df.empty:

        print()
        print("Tidak ada artikel yang ditemukan.")

        return

    # REMOVE DUPLICATES

    df = df.drop_duplicates(
        subset=["url"]
    )

    # SORT

    df = df.sort_values(
        by=["source", "category"]
    )

    # SAVE CSV

    df.to_csv(
        OUTPUT_FILE,
        index=False,
        encoding="utf-8-sig"
    )

    print()
    print("=" * 70)
    print("SCRAPING SELESAI")
    print("=" * 70)

    print(f"Total artikel : {len(df)}")
    print(f"Output        : {OUTPUT_FILE}")

    print()
    print(df[
        [
            "source",
            "category",
            "title",
            "date",
            "keywords",
        ]
    ].head(20))


if __name__ == "__main__":
    main()

==> Scraping category 'otomotif' from inews: https://www.inews.id/otomotif
  -> Fetching category page 1: https://www.inews.id/otomotif


[2026-10-01 08:52:59] INFO: Fetched (200) <GET https://www.inews.id/otomotif> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.inews.id/otomotif> (referer: https://www.google.com/)


     Found 36 potential article links.
  -> Scraping: https://www.inews.id/otomotif/motor


[2026-10-01 08:52:59] INFO: Fetched (200) <GET https://www.inews.id/otomotif/motor> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.inews.id/otomotif/motor> (referer: https://www.google.com/)


  -> Scraping: https://www.inews.id/otomotif/mobil


...

  -> Scraping: https://www.inews.id/otomotif/mobil/muncul-bintik-karat-kecil-di-bodi-mobil-awas-jangan-anggap-sepele-bisa-menyebar


[2026-10-01 08:53:33] INFO: Fetched (200) <GET https://www.inews.id/otomotif/aksesoris/pameran-komponen-otomotif-automechanika-2026-digelar-di-indonesia-diikuti-339-eksibitor> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.inews.id/otomotif/aksesoris/pameran-komponen-otomotif-automechanika-2026-digelar-di-indonesia-diikuti-339-eksibitor> (referer: https://www.google.com/)


  -> Scraping: https://www.inews.id/otomotif/aksesoris/pameran-komponen-otomotif-automechanika-2026-digelar-di-indonesia-diikuti-339-eksibitor


[2026-10-01 08:53:34] INFO: Fetched (200) <GET https://www.inews.id/otomotif/aksesoris/tren-audio-mobil-berubah-di-era-ev-intip-perkembangannya-di-indonesia> (referer: https://www.google.com/)
INFO:scrapling:Fetched (200) <GET https://www.inews.id/otomotif/aksesoris/tren-audio-mobil-berubah-di-era-ev-intip-perkembangannya-di-indonesia> (referer: https://www.google.com/)


  -> Scraping: https://www.inews.id/otomotif/aksesoris/tren-audio-mobil-berubah-di-era-ev-intip-perkembangannya-di-indonesia
<== Finished scraping category 'otomotif'. Scraped 30 articles.

SCRAPING SELESAI
Total artikel : 27
Output        : news.csv

   source  category                                              title  \
0   inews  otomotif               Berita Motor Hari Ini - iNews Portal   
1   inews  otomotif        Berita Harian Otomotif Mobil - iNews Portal   
2   inews  otomotif  Berita Pelengkap Otomotif Mobil dan Motor - iN...   
3   inews  otomotif  Jual Beli Motor dan Mobil Baru dan Bekas - iNe...   
4   inews  otomotif  <h1 class="headerTitle">200 Mobil Modifikasi B...   
5   inews  otomotif  <h1 class="headerTitle">Uji Kekuatan, Tata Kub...   
6   inews  otomotif  <h1 class="headerTitle">Perkuat Pasar Otomotif...   
7   inews  otomotif  <h1 class="headerTitle">Dikepung Mobil China, ...   
8   inews  otomotif  <h1 class="headerTitle">Fernando Alonso Jadi P...   
9   inew